In [1]:
# 셀 2: 서버 기본 세팅
import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
import threading

nest_asyncio.apply()  # Jupyter에서 asyncio 충돌 방지
app = FastAPI()

@app.get("/health")
def health():
    return {"status": "ok", "message": "서버 정상 작동 중"}
# 백그라운드에서 서버 실행
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()
print("서버 시작! http://localhost:8000/health 에서 확인")

서버 시작! http://localhost:8000/health 에서 확인


In [2]:
# 셀 3: TTS API 추가
from gtts import gTTS
import os

@app.post("/tts")
def text_to_speech(text: str):
    """
    외부 서버에서 텍스트를 보내면 → 음성 파일(.mp3)로 변환 후 반환
    Android 앱이 이 파일을 받아서 글래스 스피커로 재생
    """
    tts = gTTS(text=text, lang='ko')
    output_path = "/tmp/output_audio.mp3"
    tts.save(output_path)
    return FileResponse(
        output_path,
        media_type="audio/mpeg",
        filename="response.mp3"
    )

print("TTS API 등록 완료!")
print("테스트: POST /tts?text=역무원 도움이 필요하십니까")

TTS API 등록 완료!
테스트: POST /tts?text=역무원 도움이 필요하십니까


In [3]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from PIL import Image
import io
import torch

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
LORA_PATH = "/home/cskim/guider_model_v1/checkpoint-114"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
print("파인튜닝 모델(v1) 로드 완료!")

def analyze_with_llama(situation_text: str, target_lang: str = "ko") -> str:
    lang_names = {
        "de": "German", "fr": "French", "ja": "Japanese",
        "zh": "Chinese", "es": "Spanish", "en": "English", "ko": "Korean"
    }
    lang_name = lang_names.get(target_lang, "English")

    # RAG로 관련 매뉴얼 검색 (STEP 3 실행 후 사용 가능)
    try:
        context = search_rag(situation_text)
        print(f"[RAG] 검색 완료")
    except:
        context = ""

    if target_lang == "ko":
        prompt = f"""아래 참고 자료를 바탕으로 답변하세요.

{context}
현재 상황: {situation_text}"""
        system = "당신은 지하철 역무원 보조 AI입니다. 아래 참고 자료를 바탕으로 반드시 존댓말로 3문장 이내로 답변하세요. 참고 자료에 반말이 있어도 반드시 존댓말로 변환해서 답변하세요."
    else:
        prompt = f"""Based on the reference below, please answer.

{context}
Current situation: {situation_text}

Please answer in {lang_name}."""
        system = f"You are a subway station assistant AI. Based on the reference material, answer concisely in {lang_name} within 3 sentences."

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        input_ids,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )
    return response

@app.post("/analyze-frame")
async def analyze_frame(file: UploadFile = File(...)):
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes))
    mock_situation = f"해상도 {image.size}의 카메라 영상 수신됨. 승강장 혼잡 상황."
    analysis = analyze_with_llama(mock_situation)
    return {"analysis": analysis, "image_size": image.size}

print("카메라 프레임 분석 API 등록 완료!")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

We've detected an older driver with an RTX 4000 series GPU. These drivers have issues with P2P. This can affect the multi-gpu inference when using accelerate device_map.Please make sure to update your driver to the latest version which resolves this.


파인튜닝 모델(v1) 로드 완료!
카메라 프레임 분석 API 등록 완료!


In [4]:
# 카메라 프레임 Mock 테스트
from PIL import Image
import io, requests

# 더미 이미지 생성 (카메라 프레임 역할)
img = Image.new("RGB", (640, 480), color=(100, 100, 100))
buf = io.BytesIO()
img.save(buf, format="JPEG")
buf.seek(0)

response = requests.post(
    "http://localhost:8000/analyze-frame",
    files={"file": ("frame.jpg", buf, "image/jpeg")}
)
print("분석 결과:", response.json())

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


INFO:     127.0.0.1:55552 - "POST /analyze-frame HTTP/1.1" 200 OK
분석 결과: {'analysis': '현재 승강장 혼잡 상태로 인해 화면이 흐려 보입니다. 해상도 업그레이드 시 승객 흐름을 더 명확히 파악할 수 있을 것입니다. 승강장 혼잡 상황이 지속되면 화면 해상도 업그레이드 요청을 역장에게 알려주세요.', 'image_size': [640, 480]}


In [5]:
# 셀 7: Mock 이미지로 전체 흐름 테스트
from PIL import Image
import io, requests

# 테스트용 더미 이미지 생성
img = Image.new("RGB", (640, 480), color=(100, 100, 100))
buf = io.BytesIO()
img.save(buf, format="JPEG")
buf.seek(0)

# 1) 카메라 프레임 분석 요청
res = requests.post(
    "http://localhost:8000/analyze-frame",
    files={"file": ("frame.jpg", buf, "image/jpeg")}
)
result = res.json()
print("Llama 분석 결과:", result["analysis"])

# 2) 분석 결과를 TTS로 변환
tts_res = requests.post(
    "http://localhost:8000/tts",
    params={"text": result["analysis"]}
)
with open("/tmp/final_audio.mp3", "wb") as f:
    f.write(tts_res.content)

Audio("/tmp/final_audio.mp3")  # 전체 파이프라인 음성 확인!

INFO:     127.0.0.1:55558 - "POST /analyze-frame HTTP/1.1" 200 OK
Llama 분석 결과: 현재 승강장 혼잡 상황이 심해져서 승객 이동이 원활하지 않으셔서요. 혼잡도에 따라 승강장 입장 시간이 조절되어 있으니 참고하시길 바랍니다. 혼잡 상황이 계속되시면 역무원이 안내를 도와드릴게요.
INFO:     127.0.0.1:38582 - "POST /tts?text=%ED%98%84%EC%9E%AC+%EC%8A%B9%EA%B0%95%EC%9E%A5+%ED%98%BC%EC%9E%A1+%EC%83%81%ED%99%A9%EC%9D%B4+%EC%8B%AC%ED%95%B4%EC%A0%B8%EC%84%9C+%EC%8A%B9%EA%B0%9D+%EC%9D%B4%EB%8F%99%EC%9D%B4+%EC%9B%90%ED%99%9C%ED%95%98%EC%A7%80+%EC%95%8A%EC%9C%BC%EC%85%94%EC%84%9C%EC%9A%94.+%ED%98%BC%EC%9E%A1%EB%8F%84%EC%97%90+%EB%94%B0%EB%9D%BC+%EC%8A%B9%EA%B0%95%EC%9E%A5+%EC%9E%85%EC%9E%A5+%EC%8B%9C%EA%B0%84%EC%9D%B4+%EC%A1%B0%EC%A0%88%EB%90%98%EC%96%B4+%EC%9E%88%EC%9C%BC%EB%8B%88+%EC%B0%B8%EA%B3%A0%ED%95%98%EC%8B%9C%EA%B8%B8+%EB%B0%94%EB%9E%8D%EB%8B%88%EB%8B%A4.+%ED%98%BC%EC%9E%A1+%EC%83%81%ED%99%A9%EC%9D%B4+%EA%B3%84%EC%86%8D%EB%90%98%EC%8B%9C%EB%A9%B4+%EC%97%AD%EB%AC%B4%EC%9B%90%EC%9D%B4+%EC%95%88%EB%82%B4%EB%A5%BC+%EB%8F%84%EC%99%80%EB%93%9C%EB%A6%B4%EA%B2%8C%EC%9A%94. HTTP/1.1" 200

NameError: name 'Audio' is not defined

# 여기부터 음성 -> 텍스트 (STT)

In [4]:
# 셀 8: Whisper 모델 로드
import whisper

stt_model = whisper.load_model("small")  # 학교 서버 사양에 따라 "tiny"도 가능
print("Whisper STT 모델 로드 완료!")

Whisper STT 모델 로드 완료!


In [5]:
# 셀 9 교체: librosa로 ffmpeg 우회
import librosa
import numpy as np

def transcribe_audio(audio_path: str) -> str:
    """ffmpeg 없이 librosa로 오디오 로드 후 Whisper에 직접 전달"""
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    audio = audio.astype(np.float32)
    result = stt_model.transcribe(audio, language="ko")
    return result["text"]

@app.post("/stt")
async def speech_to_text(file: UploadFile = File(...)):
    audio_bytes = await file.read()
    audio_path = "/tmp/input_audio.wav"
    with open(audio_path, "wb") as f:
        f.write(audio_bytes)
    text = transcribe_audio(audio_path)
    return {"text": text}

print("STT API 등록 완료!")

STT API 등록 완료!


In [6]:
# 새 셀: WebSocket 서버 추가
import asyncio
import websockets
import urllib.parse
from gtts import gTTS

connected_clients = set()  # 연결된 Android 앱 목록

async def websocket_handler(websocket):
    """Android 앱이 WebSocket으로 연결되면 등록"""
    connected_clients.add(websocket)
    print(f"[WebSocket] Android 앱 연결됨! 현재 연결 수: {len(connected_clients)}")
    try:
        async for message in websocket:
            print(f"[WebSocket] 앱으로부터 메시지 수신: {message}")
    except websockets.exceptions.ConnectionClosed:
        print("[WebSocket] Android 앱 연결 끊김")
    finally:
        connected_clients.discard(websocket)

async def send_to_glass(text: str):
    """Llama 답변 텍스트를 연결된 Android 앱으로 전송"""
    if connected_clients:
        import json
        message = json.dumps({"message": text})
        await asyncio.gather(*[c.send(message) for c in connected_clients])
        print(f"[WebSocket] 전송 완료: {text}")
    else:
        print("[WebSocket] 연결된 Android 앱 없음 → Jupyter에서 직접 재생")
        tts = gTTS(text=text, lang='ko')
        tts.save("/tmp/ws_output.mp3")
        display(Audio("/tmp/ws_output.mp3", autoplay=True))

# WebSocket 서버 백그라운드 실행 (포트 8765)
async def start_ws_server():
    async with websockets.serve(websocket_handler, "0.0.0.0", 8765):
        print("[WebSocket] 서버 시작! 포트 8765")
        await asyncio.Future()

import threading
def run_ws():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(start_ws_server())

ws_thread = threading.Thread(target=run_ws, daemon=True)
ws_thread.start()
print("WebSocket 서버 시작 완료!")

WebSocket 서버 시작 완료!


In [ ]:
import os
os.system("fuser -k 8000/tcp")
os.system("fuser -k 8765/tcp")
print("포트 해제 완료!")

In [7]:
# 셀 12: gTTS 대신 wav 파일로 Mock 음성 생성
import numpy as np
import soundfile as sf

# 실제 음성 대신 빈 wav 파일 생성 (Whisper 파이프라인 테스트용)
sample_rate = 16000
duration = 3  # 초
audio_data = np.zeros(sample_rate * duration, dtype=np.float32)
sf.write("/tmp/mock_voice_input.wav", audio_data, sample_rate)

print("Mock wav 파일 생성 완료!")

Mock wav 파일 생성 완료!


In [8]:
import requests
response = requests.get("http://localhost:8000/health")
print(response.json())

INFO:     127.0.0.1:49584 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok', 'message': '서버 정상 작동 중'}


In [9]:
from IPython.display import Audio

response = requests.post(
    "http://localhost:8000/tts",
    params={"text": "안녕하세요. 역무원 AI 보조 시스템입니다."}
)
with open("/tmp/tts_test.mp3", "wb") as f:
    f.write(response.content)

Audio("/tmp/tts_test.mp3")

INFO:     127.0.0.1:49590 - "POST /tts?text=%EC%95%88%EB%85%95%ED%95%98%EC%84%B8%EC%9A%94.+%EC%97%AD%EB%AC%B4%EC%9B%90+AI+%EB%B3%B4%EC%A1%B0+%EC%8B%9C%EC%8A%A4%ED%85%9C%EC%9E%85%EB%8B%88%EB%8B%A4. HTTP/1.1" 200 OK


In [10]:
import numpy as np
import soundfile as sf

audio_data = np.zeros(16000 * 3, dtype=np.float32)
sf.write("/tmp/test.wav", audio_data, 16000)

with open("/tmp/test.wav", "rb") as f:
    response = requests.post(
        "http://localhost:8000/conversation",
        files={"file": ("test.wav", f, "audio/wav")}
    )
print(response.json())

INFO:     127.0.0.1:47576 - "POST /conversation HTTP/1.1" 404 Not Found
{'detail': 'Not Found'}


In [ ]:
from PIL import Image
import io

img = Image.new("RGB", (640, 480), color=(100, 100, 100))
buf = io.BytesIO()
img.save(buf, format="JPEG")
buf.seek(0)

response = requests.post(
    "http://localhost:8000/analyze-frame",
    files={"file": ("frame.jpg", buf, "image/jpeg")}
)
print(response.json())

In [12]:
# 셀 14
import urllib.parse
from IPython.display import Audio, display
import os

@app.post("/conversation")
async def conversation(file: UploadFile = File(...)):
    # 1) 음성 → 텍스트 + 언어 감지
    audio_bytes = await file.read()
    import os
    ext = os.path.splitext(file.filename)[1]
    if not ext:
        ext = ".m4a"
    audio_path = f"/tmp/conv_input{ext}"
    with open(audio_path, "wb") as f:
        f.write(audio_bytes)
    print(f"저장된 파일: {audio_path}")  # 확인용

    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    audio = audio.astype(np.float32)

    # 언어 먼저 감지
    detect_result = stt_model.transcribe(audio, language=None, task="transcribe")
    detected_lang = detect_result["language"]
    print(f"[STT] 감지된 언어: {detected_lang}")

    # 한국어면 그대로, 외국어면 영어로 변환
    if detected_lang == "ko":
        user_text = detect_result["text"]
        print(f"[STT] 한국어 원문: {user_text}")
    else:
        en_result = stt_model.transcribe(audio, language=None, task="translate")
        user_text = en_result["text"]
        print(f"[STT] 영어 변환: {user_text}")

    # 2) Llama 답변 생성
    answer = analyze_with_llama(user_text, detected_lang)
    print(f"[Llama] 답변: {answer}")

    # 3) Android 앱 연결 여부에 따라 자동 분기
    await send_to_glass(answer)

    return {
        "status": "ok",
        "detected_language": detected_lang,
        "stt_text": user_text,
        "llm_answer": answer
    }

print("통합 대화 API (다국어 지원) 등록 완료!")

통합 대화 API (다국어 지원) 등록 완료!


In [ ]:
import numpy as np
import soundfile as sf
import requests
from gtts import gTTS

# 영어 음성 파일 생성 (테스트용)
tts = gTTS(text="Where is the Toilet?", lang='en')
tts.save("/tmp/english_test.mp3")

# mp3를 wav로 변환
import librosa
audio, sr = librosa.load("/tmp/english_test.mp3", sr=16000, mono=True)
sf.write("/tmp/english_test.wav", audio, sr)

# /conversation API로 전송
with open("/tmp/english_test.wav", "rb") as f:
    response = requests.post(
        "http://localhost:8000/conversation",
        files={"file": ("english_test.wav", f, "audio/wav")}
    )

print(response.json())

In [ ]:
import numpy as np
import soundfile as sf
import requests
from gtts import gTTS
import librosa

tts = gTTS(text="Wo ist die Toilette?", lang='de')
tts.save("/tmp/german_test.mp3")

audio, sr = librosa.load("/tmp/german_test.mp3", sr=16000, mono=True)
sf.write("/tmp/german_test.wav", audio, sr)

with open("/tmp/german_test.wav", "rb") as f:
    response = requests.post(
        "http://localhost:8000/conversation",
        files={"file": ("german_test.wav", f, "audio/wav")}
    )

result = response.json()
print("감지 언어:", result.get("detected_language"))
print("STT:", result.get("stt_text"))
print("답변:", result.get("llm_answer"))

In [14]:
from sentence_transformers import SentenceTransformer
import faiss
import json
import numpy as np

embed_model = SentenceTransformer("snunlp/KR-SBERT-V40K-klueNLI-augSTS")
print("임베딩 모델 로드 완료!")

data = []
with open("/home/cskim/train_data_standardized.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        data.append({"question": item["question"], "answer": item["answer"]})

print(f"데이터 로드 완료: {len(data)}개")

# 벡터화 + 인덱스 생성
questions = [d["question"] for d in data]
embeddings = embed_model.encode(questions, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print(f"FAISS 인덱스 재생성 완료! 총 {index.ntotal}개 벡터")

/home/cskim/.local/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


임베딩 모델 로드 완료!
데이터 로드 완료: 338개


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

FAISS 인덱스 재생성 완료! 총 338개 벡터


In [19]:
def search_rag(query: str, top_k: int = 3) -> str:
    import re
    
    # ① 숫자 코드 추출 (예: 531, 535)
    code_match = re.search(r'\d{3,}', query)
    
    if code_match:
        code_num = code_match.group()
        
        # 숫자 코드가 포함된 항목 직접 검색
        keyword_results = [
            item for item in data 
            if code_num in item["question"]
        ]
        
        if keyword_results:
            context = ""
            for i, item in enumerate(keyword_results[:top_k]):
                context += f"[참고 {i+1}]\n"
                context += f"질문: {item['question']}\n"
                context += f"답변: {item['answer']}\n\n"
            print(f"[RAG] 키워드 매칭 ({code_num}) {len(keyword_results[:top_k])}개 발견!")
            return context
    
    # ② 숫자 코드 없으면 기존 벡터 유사도 검색
    query_vec = embed_model.encode([query]).astype("float32")
    distances, indices = index.search(query_vec, top_k)
    
    context = ""
    for i, idx in enumerate(indices[0]):
        context += f"[참고 {i+1}]\n"
        context += f"질문: {data[idx]['question']}\n"
        context += f"답변: {data[idx]['answer']}\n\n"
    
    print(f"[RAG] 벡터 유사도 검색 {top_k}개 반환")
    return context

print("search_rag 키워드 검색 추가 완료!")

search_rag 키워드 검색 추가 완료!


In [16]:
import numpy as np
import soundfile as sf
import requests
from gtts import gTTS
import librosa

# 501 장애 코드 질문 음성 생성
tts = gTTS(text="PSD 501 장애 코드가 떴는데 어떻게 해야 하나요?", lang='ko')
tts.save("/tmp/psd_test.mp3")

audio, sr = librosa.load("/tmp/psd_test.mp3", sr=16000, mono=True)
sf.write("/tmp/psd_test.wav", audio, sr)

with open("/tmp/psd_test.wav", "rb") as f:
    response = requests.post(
        "http://localhost:8000/conversation",
        files={"file": ("psd_test.wav", f, "audio/wav")}
    )

result = response.json()
print("감지 언어:", result.get("detected_language"))
print("STT:", result.get("stt_text"))
print("답변:", result.get("llm_answer"))

저장된 파일: /tmp/conv_input.wav


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[STT] 감지된 언어: ko
[STT] 한국어 원문:  PSD-501 장애 코드가 떴는데 어떻게 해야하나요?
[RAG] 검색 완료
[Llama] 답변: 501번은 '하부 가이드 레일 이물질 끼임'이야. 승객 안전 확인부터 하고, 발판 사이를 점검해서 큰 이물질이 있으면 수동으로 제거해. 열차 출발 전에 해결 못 하면 관제실 보고해서 해당 문 잠금 처리해. 혼잡 상황이면 다른 역무원 긴급 지원 요청하고, 장애 문 양옆의 정상 문으로 승객 유도해서 혼잡 상황 통제해. 방송으로 '○번 문 잠시 이용 불가, 옆 문 이용 바랍니다' 안내해. 혼잡 상황이면 2인 1조 대응이 원칙이야. 관제실에 보고해서 해당
[WebSocket] 연결된 Android 앱 없음 → Jupyter에서 직접 재생


INFO:     127.0.0.1:54456 - "POST /conversation HTTP/1.1" 200 OK
감지 언어: ko
STT:  PSD-501 장애 코드가 떴는데 어떻게 해야하나요?
답변: 501번은 '하부 가이드 레일 이물질 끼임'이야. 승객 안전 확인부터 하고, 발판 사이를 점검해서 큰 이물질이 있으면 수동으로 제거해. 열차 출발 전에 해결 못 하면 관제실 보고해서 해당 문 잠금 처리해. 혼잡 상황이면 다른 역무원 긴급 지원 요청하고, 장애 문 양옆의 정상 문으로 승객 유도해서 혼잡 상황 통제해. 방송으로 '○번 문 잠시 이용 불가, 옆 문 이용 바랍니다' 안내해. 혼잡 상황이면 2인 1조 대응이 원칙이야. 관제실에 보고해서 해당


In [18]:
# 파인튜닝 확인 테스트 코드
result = analyze_with_llama("PSD 장애코드 501")
print(result)

[RAG] 검색 완료
501번은 '하부 가이드레일 이물질 끼임'이야. 승객 안전 확인부터 하고, 발판 사이를 점검해봐. 큰 이물질이면 수동으로 제거해야 해. 절대 열차 출발 전에 해결 못 하면 관제실 보고하고 해당 문 잠금 처리해. 안전에 유의해서 수동 제거 작업을 해줘. 관제실에 수동 제거 완료 후 다시 열차 출발 가능 여부를 확인해줘.


In [15]:
# 531 데이터 있는지 직접 확인
results = [d for d in data if "531" in d["question"]]
print(f"531 관련 데이터: {len(results)}개")
for r in results:
    print(r["question"])
    print(r["answer"])
    print("---")

531 관련 데이터: 1개
PSD 531 코드 처음 봤어. 뭐야?
531은 '비상 개방 후 미복구' 코드야. 비상 개방이 실행됐는데 정상 잠금 상태로 돌아오지 않은 거야. 수동 복구 키로 잠금 상태 확인하고, 안 되면 해당 문 비활성화 처리 후 시설팀 긴급 출동 요청해. 비상 개방 이력은 반드시 일지에 기록해.
---


In [25]:
# 531 코드 검색 테스트
context = search_rag("PSD 531번 코드가 뭐야?")
print(context)

[참고 1]
질문: PSD 531 코드가 뭐야?
답변: 531은 '비상 개방 후 미복구' 코드야. 비상 개방이 실행됐는데 정상 잠금 상태로 돌아오지 않은 거야. 수동 복구 키로 잠금 상태 확인하고, 안 되면 해당 문 비활성화 처리 후 시설팀 긴급 출동 요청해. 비상 개방 이력은 반드시 일지에 기록해.

[참고 2]
질문: PSD 532 코드가 뭐야?
답변: 532는 '도어 센서 오염 감지' 코드야. 센서 렌즈에 먼지나 이물질이 붙어 오작동하는 거야. 마른 면 천으로 센서 렌즈를 부드럽게 닦아봐. 그래도 지속되면 센서 교체가 필요하니까 시설팀 통보해. 당분간 해당 문 수동 감시 모드로 운영해.

[참고 3]
질문: PSD 533 코드가 뭐야?
답변: 533은 '개폐 속도 이상' 코드야. 문이 너무 느리거나 빠르게 움직이는 경우야. 모터 구동부 이상 또는 레일 마찰 과다가 원인일 수 있어. 승객 끼임 위험이 있으니 해당 문 일시 비활성화하고, 시설팀에 점검 요청해. 속도 이상은 방치하면 모터 과부하로 이어질 수 있어.




In [17]:
query_vec = embed_model.encode(["PSD 535번 코드가 뭐야?"]).astype("float32")
distances, indices = index.search(query_vec, 5)

for dist, idx in zip(distances[0], indices[0]):
    print(f"거리: {dist:.4f} / {data[idx]['question']}")

거리: 38.4550 / PSD 534 코드가 뭐야?
거리: 38.6956 / PSD 535 코드가 뭐야?
거리: 39.9286 / PSD 532 코드가 뭐야?
거리: 40.7718 / PSD 533 코드가 뭐야?
거리: 43.8795 / PSD 531 코드가 뭐야?


In [18]:
# 535 직접 테스트
context = search_rag("PSD 535번 코드가 뭐야?")
print(context)

# Llama 답변도 확인
result = analyze_with_llama("PSD 535번 코드가 뭐야?", "ko")
print(result)

[참고 1]
질문: PSD 534 코드가 뭐야?
답변: 534는 '배터리 백업 전압 저하' 코드야. 정전 시 비상 개방을 위한 배터리가 방전 위기라는 신호야. 당장 운영에 지장은 없지만 정전이 나면 비상 개방이 안 될 수 있어. 즉시 시설팀에 배터리 교체 요청하고, 조치 완료 전까지 해당 구역 수동 비상 대응 체계를 유지해.

[참고 2]
질문: PSD 535 코드가 뭐야?
답변: 535는 '중앙 제어 통신 두절' 코드야. 개별 PSD 유닛이 중앙 관제 시스템과 연결이 끊긴 거야. 해당 문은 독립 모드로 전환돼서 중앙 제어가 안 돼. 관제실에 즉시 보고하고, 해당 문은 역무원이 직접 현장에서 수동 제어해야 해. 통신 복구 전까지 해당 승강장 상주 인원 배치해.

[참고 3]
질문: PSD 532 코드가 뭐야?
답변: 532는 '도어 센서 오염 감지' 코드야. 센서 렌즈에 먼지나 이물질이 붙어 오작동하는 거야. 마른 면 천으로 센서 렌즈를 부드럽게 닦아봐. 그래도 지속되면 센서 교체가 필요하니까 시설팀 통보해. 당분간 해당 문 수동 감시 모드로 운영해.


[RAG] 검색 완료
중앙 제어 통신 두절이야. 해당 문은 역무원이 직접 현장에서 수동 제어해야 해. 통신 복구 전까지 해당 승강장 상주 인원이 배치되어야 해. 관제실에 즉시 보고해서 통신 복구 시까지 수동 운영을 유지해. 역무원이 직접 현장 수동 제어를 하기 때문에 해당 역에 역무원이 충분히 배치되어 있어야 해. 역무원이 부족하면 수동 운영이 어려워. 즉시 시설팀에 통신 복구 요청하고, 복구 전까지 해당 문은 역무원이 직접 수동으로 관리해. 역장과 승강장 역무


In [19]:
# 질문 형식 확인
for item in data:
    print(item["question"])

선배님, PSD 장애 코드 501이 떴는데 어떻게 하죠?
PSD 502 장애 코드가 뭐야?
PSD 503 코드 떴어. 뭔데 이거?
비상문이 갑자기 열렸어. 어떻게 해?
PSD 장애물 감지 센서가 계속 오작동하는 것 같아. 어떻게 해야 해?
PSD가 열차와 동기화가 안 되는 것 같아. 문이 안 열려.
PSD 504 코드가 뭐야?
PSD 전원이 갑자기 나갔어!
승객이 PSD와 열차 사이에 끼였어!
PSD 505 코드는 뭐야?
PSD 문이 반만 열려. 완전히 안 열려.
PSD 수동 해제 키가 어디 있어?
PSD 506 코드 떴어.
PSD 열림 신호는 떴는데 실제로 문이 안 열려. 뭐가 문제야?
PSD 점검은 보통 언제 하는 거야?
PSD 507 코드가 떴는데 처음 보는 코드야.
러시아워 때 PSD 장애가 나면 어떻게 대처해야 해?
PSD 508 코드는 뭐야?
PSD 장애 발생 시 승객한테는 어떻게 안내해야 해?
PSD 509 코드 떴어.
PSD 비상 개방 절차 전체를 알려줘.
PSD 510 코드 처음 봤어.
여름철에 PSD 장애가 유독 많이 발생하는 이유가 뭐야?
PSD 511 코드가 뭐야?
PSD 장애 기록은 어디에, 어떻게 써야 해?
PSD 512 코드 떴어.
PSD와 열차 문이 동시에 장애 나면 어떻게 해?
PSD 513 코드가 뭐야?
PSD 문에 승객 가방이 끼었어. 어떻게 해?
PSD 514 코드 떴어.
PSD 점검 중 작업자 안전은 어떻게 해야 해?
PSD 515 코드 뭐야?
동절기에 PSD 결빙이 생기면 어떻게 해?
PSD 516 코드가 뭐야?
PSD 복구 후 정상 동작 확인은 어떻게 해?
PSD 517 코드 떴어.
PSD 고장이 잦은 문이 있어. 보고는 어떻게 해야 해?
PSD 518 코드 뭐야?
PSD 수동 안전바는 언제 써?
PSD 519 코드 떴어.
PSD가 자꾸 열렸다 닫혔다 반복해. 뭐가 문제야?
PSD 520 코드가 뭐야?
PSD 장애 시 열차 지연은 어떻게 처리해?
PSD 장애 시 기관사와는 어떻게 소통해야 해?
야간에 PSD 장애 나면 

In [20]:
import os
print(os.path.exists("/home/cskim/train_data_standardized.jsonl"))

True


In [21]:
import json

input_path = "/home/cskim/train_data_standardized.jsonl"
output_path = "/home/cskim/train_data_standardized_v2.jsonl"

def normalize_question(question: str) -> str:
    """질문 형식 통일: 핵심 키워드만 추출해서 '~가 뭐야?' 형식으로"""
    import re
    
    # PSD 코드 번호 추출
    psd_match = re.search(r'PSD\s*(\d+)', question)
    if psd_match:
        code_num = psd_match.group(1)
        return f"PSD {code_num} 코드가 뭐야?"
    
    # PSD 코드 아닌 경우 그대로 유지
    return question

# 변환 실행
new_data = []
changed = 0

with open(input_path, "r", encoding="utf-8-sig") as f:
    for line in f:
        item = json.loads(line)
        messages = item["messages"]
        
        user_msg = next((m["content"] for m in messages if m["role"] == "user"), "")
        assistant_msg = next((m["content"] for m in messages if m["role"] == "assistant"), "")
        
        new_question = normalize_question(user_msg)
        if new_question != user_msg:
            changed += 1
        
        new_data.append({
            "question": new_question,
            "answer": assistant_msg
        })

# 새 파일로 저장
with open(output_path, "w", encoding="utf-8") as f:
    for item in new_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ 완료! 총 {len(new_data)}개 중 {changed}개 질문 형식 변경")
print(f"저장 위치: {output_path}")

✅ 완료! 총 338개 중 22개 질문 형식 변경
저장 위치: /home/cskim/train_data_standardized_v2.jsonl


In [22]:
# 변경 안 된 질문들 샘플 확인
with open(output_path, "r", encoding="utf-8") as f:
    sample_data = [json.loads(line) for line in f]

# 앞에 20개만 출력
for item in sample_data[:20]:
    print(item["question"])
    print("---")

선배님, PSD 장애 코드 501이 떴는데 어떻게 하죠?
---
PSD 502 코드가 뭐야?
---
PSD 503 코드가 뭐야?
---
비상문이 갑자기 열렸어. 어떻게 해?
---
PSD 장애물 감지 센서가 계속 오작동하는 것 같아. 어떻게 해야 해?
---
PSD가 열차와 동기화가 안 되는 것 같아. 문이 안 열려.
---
PSD 504 코드가 뭐야?
---
PSD 전원이 갑자기 나갔어!
---
승객이 PSD와 열차 사이에 끼였어!
---
PSD 505 코드가 뭐야?
---
PSD 문이 반만 열려. 완전히 안 열려.
---
PSD 수동 해제 키가 어디 있어?
---
PSD 506 코드가 뭐야?
---
PSD 열림 신호는 떴는데 실제로 문이 안 열려. 뭐가 문제야?
---
PSD 점검은 보통 언제 하는 거야?
---
PSD 507 코드가 뭐야?
---
러시아워 때 PSD 장애가 나면 어떻게 대처해야 해?
---
PSD 508 코드가 뭐야?
---
PSD 장애 발생 시 승객한테는 어떻게 안내해야 해?
---
PSD 509 코드가 뭐야?
---


In [23]:
import json
import re

input_path = "/home/cskim/train_data_standardized_v2.jsonl"
output_path = "/home/cskim/train_data_standardized_v3.jsonl"

def normalize_question(question: str) -> str:
    # PSD 코드 번호 포함된 질문 통일
    psd_match = re.search(r'PSD\s*(?:장애\s*코드\s*)?(\d+)', question)
    if psd_match:
        code_num = psd_match.group(1)
        return f"PSD {code_num} 코드가 뭐야?"
    return question

new_data = []
changed = 0

with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        new_question = normalize_question(item["question"])
        if new_question != item["question"]:
            changed += 1
        new_data.append({
            "question": new_question,
            "answer": item["answer"]
        })

with open(output_path, "w", encoding="utf-8") as f:
    for item in new_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ 완료! 총 {len(new_data)}개 중 {changed}개 추가 변경")
print(f"저장: {output_path}")

✅ 완료! 총 338개 중 1개 추가 변경
저장: /home/cskim/train_data_standardized_v3.jsonl


In [13]:
data = []
with open("/home/cskim/train_data_standardized.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        data.append({"question": item["question"], "answer": item["answer"]})

print(f"데이터 로드 완료: {len(data)}개")

# 벡터화 + 인덱스 생성
questions = [d["question"] for d in data]
embeddings = embed_model.encode(questions, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print(f"FAISS 인덱스 재생성 완료! 총 {index.ntotal}개 벡터")

데이터 로드 완료: 338개


NameError: name 'embed_model' is not defined

In [20]:
# 535 테스트
context = search_rag("PSD 535번 코드가 뭐야?")
print(context)

[RAG] 키워드 매칭 (535) 1개 발견!
[참고 1]
질문: PSD 535 코드가 뭐야?
답변: 535는 '중앙 제어 통신 두절' 코드야. 개별 PSD 유닛이 중앙 관제 시스템과 연결이 끊긴 거야. 해당 문은 독립 모드로 전환돼서 중앙 제어가 안 돼. 관제실에 즉시 보고하고, 해당 문은 역무원이 직접 현장에서 수동 제어해야 해. 통신 복구 전까지 해당 승강장 상주 인원 배치해.


